In [1]:
import random
import json
import re
from tqdm import tqdm

TOTAL_SAMPLES = 50000
output_file = "datasetgeneree.jsonl"

SHARED_SYSTEM_PROMPT = (
    "You are a professional assistant. Your role is to help clients book, cancel, or "
    "reschedule appointments with precision and politeness. Always ensure you have "
    "the service type, the day, and the specific time before confirming."
)

In [2]:
# --- 2. DICTIONNAIRES ENRICHIS (Qualité & Naturel) ---
TITLES = ["Mr.", "Ms."]
SERVICES = ["haircut", "yoga class", "car maintenance", "dental checkup", "consultation", "massage", "coaching", "legal advice"]
LOCATIONS = ["Downtown", "City Center", "West Plaza", "North Branch"]
NAMES = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis"]
DAYS = ["Monday", "next Tuesday", "Wednesday", "tomorrow", "Friday", "April 10th"]
TIMES = ["9:00 AM", "10:30 AM", "2:00 PM", "4:45 PM", "11:15 AM", "5:30 PM"]

# Variantes pour le naturel
FILLER_WORDS = ["Actually,", "Let me see...", "Mhm,", "I understand.", "Sure thing!"]
POLITE_CLOSINGS = ["See you then!", "Have a great day!", "We look forward to seeing you.", "Take care!"]


In [3]:
# --- 3. LOGIQUE DE SCÉNARIO AVANCÉE (Gestion des données manquantes) ---
def generate_scenario(intent):
    s, d, n = random.choice(SERVICES), random.choice(DAYS), random.choice(NAMES)
    t1, t2 = random.sample(TIMES, 2)
    title, loc = random.choice(TITLES), random.choice(LOCATIONS)

    flow = []

    if intent == "booking":
        # Tour 1 : Salutation
        flow.append({"role": "assistant", "content": "Hello! How can I help you today?"})

        # Tour 2 : L'utilisateur donne une info incomplète
        r = random.random()
        if r < 0.3:
            flow.append({"role": "user", "content": f"I'd like to book a {s}"})
            # L'IA doit demander le JOUR (Manquant)
            flow.append({"role": "assistant", "content": f"{random.choice(FILLER_WORDS)} I can help with that {s}. Which day would you like to come in?"})
            flow.append({"role": "user", "content": f"Can we do {d}?"})
        else:
            # L'utilisateur donne service + jour, mais pas l'HEURE
            flow.append({"role": "user", "content": f"Hi, I need an appointment for a {s} on {d}."})

        # Tour 3 : L'IA PROPOSE 2 CRÉNEAUX (Réponse au besoin "Ask for time")
        flow.append({"role": "assistant", "content": f"Perfect. For {d}, I have two slots available: {t1} or {t2}. Which one works best for you?"})

        # Tour 4 : Choix de l'utilisateur
        flow.append({"role": "user", "content": f"I'll take the {t1} slot, please."})

        # Tour 5 : Demande du nom (Finalisation)
        flow.append({"role": "assistant", "content": "Great choice. To finalize the booking, may I have your last name?"})
        flow.append({"role": "user", "content": f"My name is {n}."})

        # Tour 6 : Confirmation avec MÉMOIRE (Récapitulatif complet)
        flow.append({"role": "assistant", "content": f"It's noted {title} {n}. Your {s} is confirmed at {loc} for {d} at {t1}. {random.choice(POLITE_CLOSINGS)}"})

    elif intent == "rescheduling":
        flow.append({"role": "assistant", "content": "Welcome! How can I assist you?"})
        flow.append({"role": "user", "content": f"I have a {s} on {d} but I need to change it."})
        flow.append({"role": "assistant", "content": f"No problem. I can move that. I have {t1} or {t2} available. Which one do you prefer?"})
        flow.append({"role": "user", "content": f"Let's go with {t1}."})
        flow.append({"role": "assistant", "content": f"All set {title} {n}. I've updated your {s} to {t1} on {d}. {random.choice(POLITE_CLOSINGS)}"})

    elif intent == "cancellation":
        flow.append({"role": "assistant", "content": "Hello, how can I help?"})
        flow.append({"role": "user", "content": f"I need to cancel my {s}."})
        flow.append({"role": "assistant", "content": f"I'm sorry to hear that. Could you remind me which day it was for?"})
        flow.append({"role": "user", "content": f"It was for {d}."})
        flow.append({"role": "assistant", "content": f"Thank you {title} {n}. Your {s} for {d} has been removed from our schedule. {random.choice(POLITE_CLOSINGS)}"})

    else: # information
        flow.append({"role": "assistant", "content": "Hi! How can I help?"})
        flow.append({"role": "user", "content": f"Where is your {loc} office?"})
        flow.append({"role": "assistant", "content": f"Our {loc} branch is located right next to the main square. Anything else?"})
        flow.append({"role": "user", "content": "No, that's all. Thanks!"})
        flow.append({"role": "assistant", "content": f"You're welcome! {random.choice(POLITE_CLOSINGS)}"})

    return flow


In [4]:
# --- 4. BOUCLE DE GÉNÉRATION ---
synthetic_dataset = []
INTENTS = ["booking", "cancellation", "rescheduling", "information"]
samples_per_intent = TOTAL_SAMPLES // len(INTENTS)

for intent in INTENTS:
    print(f"Generating {samples_per_intent} examples for: {intent}...")
    for _ in tqdm(range(samples_per_intent)):
        conv = generate_scenario(intent)
        synthetic_dataset.append({
            "messages": [{"role": "system", "content": SHARED_SYSTEM_PROMPT}] + conv
        })

# Mélange final
random.shuffle(synthetic_dataset)

Generating 12500 examples for: booking...


100%|██████████| 12500/12500 [00:00<00:00, 123583.17it/s]


Generating 12500 examples for: cancellation...


100%|██████████| 12500/12500 [00:00<00:00, 138392.99it/s]


Generating 12500 examples for: rescheduling...


100%|██████████| 12500/12500 [00:00<00:00, 134417.31it/s]


Generating 12500 examples for: information...


100%|██████████| 12500/12500 [00:00<00:00, 118907.20it/s]


In [5]:
# --- 5. SAUVEGARDE ---
with open(output_file, "w", encoding="utf-8") as f:
    for entry in synthetic_dataset:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\n✅ DONE! {len(synthetic_dataset)} conversations generated in {output_file}")


✅ DONE! 50000 conversations generated in datasetgeneree.jsonl
